# Calculatrice seq2seq

Dans ces travaux pratiques, un réseau récurrent apprend à calculer à partir de texte : il lit une équation écrite caractère par caractère, comme `12+34`, et écrit son résultat, `46`, caractère par caractère lui aussi. C'est un modèle *sequence to sequence* (seq2seq) : un encodeur résume la séquence d'entrée, un décodeur produit la séquence de sortie.

1. Encoder les équations.
2. Générer le jeu de données.
3. Construire l'encodeur-décodeur.
4. L'entraîner, puis l'évaluer.
5. La réussite par opération.
6. Faut-il inverser l'équation ?

Ce TP a besoin d'une carte graphique : vérifiez dans `Exécution > Modifier le type d'exécution` que l'accélérateur matériel est un GPU. Sur un processeur, ses deux entraînements prendraient plusieurs heures.

## Installation et import des bibliothèques

In [ ]:
!pip install -q lightning

In [ ]:
import operator
import pathlib
from collections.abc import Sequence

import lightning
import matplotlib.pyplot as plt
import numpy as np
import pandas
import sklearn.model_selection
import torch
from lightning.pytorch.callbacks import Callback, TQDMProgressBar
from lightning.pytorch.loggers import CSVLogger
from lightning.pytorch.utilities.model_summary import ModelSummary
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

# Mêmes tirages aléatoires à chaque exécution : des résultats reproductibles
lightning.seed_everything(0)

## 1. Encoder les équations

Les équations et leurs résultats s'écrivent avec 16 caractères : les chiffres, le point décimal, les 4 opérations et l'espace, qui sert de *padding*. `encode` transforme une chaîne en tenseur d'indices de ce vocabulaire, `decode` fait l'inverse.

In [ ]:
operations = list("+*-/")
nombres = list("0123456789.")
padding = [" "]

caracteres = nombres + operations + padding
indices_caracteres = {caractere: index for index, caractere in enumerate(caracteres)}


def encode(chaine: str) -> torch.Tensor:
  return torch.tensor([indices_caracteres[caractere] for caractere in chaine])


def decode(indices: torch.Tensor) -> str:
  return "".join(caracteres[int(index)] for index in indices)


print(f"Vocabulaire : {caracteres}")
print(f'encode("3/4+2-5") = {encode("3/4+2-5")}')
print(f'decode(encode("3/4+2-5")) = {decode(encode("3/4+2-5"))}')

### Quelle entrée pour le réseau ?

Le réseau peut-il travailler directement sur la sortie de `encode`, des entiers de 0 à 15, ou faut-il une transformation de plus ? Laquelle ?

Votre réponse ici

#### Solution

Ces indices désignent des catégories, pas des quantités : que `+` ait l'indice 11 et `9` l'indice 9 ne veut pas dire que `+` vaut plus que `9`. Donnés tels quels, ils feraient croire au réseau à un ordre et à des distances qui n'existent pas. Il faut donc, au choix :

- un encodage *one-hot* : un vecteur de 16 valeurs, toutes nulles sauf un 1 à l'indice du caractère ;
- une couche `nn.Embedding`, qui apprend un vecteur par caractère, comme dans les slides. C'est ce que fera notre modèle.

## 2. Générer le jeu de données

`fabrique_probleme` tire une opération au hasard et deux entiers de 0 à 999, jusqu'à obtenir le nombre d'équations demandé, toutes différentes :

- les divisions par 0 sont écartées ;
- les résultats sont tronqués à 6 caractères (`0.3333`) ;
- équations et résultats sont complétés par des espaces, pour faire tous 7 et 6 caractères ;
- l'équation est inversée par défaut (`inverse=True`) : `12+34` devient `43+21`. La partie 6 dit pourquoi.

In [ ]:
def fabrique_probleme(
  nb_exemples: int = 1000, nb_chiffres: int = 3, inverse: bool = True
) -> tuple[torch.Tensor, torch.Tensor]:
  equations = []
  resultats = []
  vues = set()

  fonction_operation = {
    "+": operator.add,
    "-": operator.sub,
    "*": operator.mul,
    "/": operator.truediv,
  }

  # Longueurs maximales de l'équation et du résultat
  longueur_max_equation = 2 * nb_chiffres + 1
  longueur_max_resultat = 2 * nb_chiffres

  while len(equations) < nb_exemples:
    operation = np.random.choice(operations)
    gauche, droite = np.random.randint(10**nb_chiffres, size=2)
    equation = f"{gauche}{operation}{droite}"
    # Une équation déjà tirée, ou une division par 0 : on recommence
    if equation in vues or (operation == "/" and droite == 0):
      continue
    vues.add(equation)

    # Un résultat comme 0.3333333… est tronqué
    resultat = str(fonction_operation[operation](gauche, droite))[
      :longueur_max_resultat
    ]
    # Des espaces pour que toutes les séquences aient la même longueur
    equation_completee = equation.ljust(longueur_max_equation)
    resultat_complete = resultat.ljust(longueur_max_resultat)
    if inverse:
      equation_completee = equation_completee[::-1]

    equations.append(equation_completee)
    resultats.append(resultat_complete)

  X = torch.stack([encode(equation) for equation in equations])
  y = torch.stack([encode(resultat) for resultat in resultats])
  return X, y


X, y = fabrique_probleme(10)
print(f"Formes de X et y : {tuple(X.shape)}, {tuple(y.shape)}")
for equation_encodee, resultat_encode in zip(X, y):
  # L'équation est inversée : on la remet dans l'ordre pour l'afficher
  print(f"{decode(equation_encodee.flip(0))} = {decode(resultat_encode)}")

Le jeu de données : 100 000 équations, dont 80 % pour l'entraînement et 20 % pour le test.

In [ ]:
X, y = fabrique_probleme(100_000)
X_train, X_test, y_train, y_test = sklearn.model_selection.train_test_split(
  X, y, train_size=0.8, random_state=0
)
print(tuple(X_train.shape), tuple(X_test.shape))

## 3. Construire l'encodeur-décodeur

Écrivez la classe `Seq2Seq`, un `nn.Module` :

- `__init__(nb_unites=512)` crée ses couches :
  - **l'encodeur** : une couche `nn.Embedding` de 16 dimensions, puis un `nn.GRU` de deux couches empilées de `nb_unites` unités ;
  - **le décodeur** : un `nn.GRU` d'une couche de `nb_unites` unités, puis une couche `nn.Linear` qui donne un score (un logit) à chacun des 16 caractères ;
- `forward(equations)` reçoit des équations encodées, de forme `(batch, 7)` (`longueur_max_equation`) :
  - l'encodeur les lit ; le dernier état caché de sa seconde couche est le résumé de toute l'équation ;
  - ce résumé est répété 6 fois (`longueur_max_resultat`), une fois par caractère du résultat, et le décodeur lit cette séquence ;
  - la couche `nn.Linear`, appliquée à la sortie du décodeur à chaque pas, donne les logits de forme `(batch, 6, 16)`.

Ce décodeur ne reçoit pas le caractère qu'il vient d'écrire : chaque pas reçoit le même résumé de l'équation. Il n'y a donc pas de *teacher forcing*, qui donne au décodeur, à l'entraînement, le vrai caractère précédent du résultat. C'est le plus simple des décodeurs : il suffit à l'addition et à la soustraction, pas à la multiplication ni à la division (partie 5).

**Documentation utile**

- [`torch.nn.GRU`](https://docs.pytorch.org/docs/stable/generated/torch.nn.GRU.html)
- [`torch.Tensor.repeat`](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.repeat.html)

In [ ]:
longueur_max_equation, longueur_max_resultat = X.shape[1], y.shape[1]


class Seq2Seq(nn.Module):
  def __init__(self, nb_unites: int = 512) -> None:
    super().__init__()
    # Votre code ici

  def forward(self, equations: torch.Tensor) -> torch.Tensor:
    # Votre code ici
    ...

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `nn.Embedding(len(caracteres), 16)`
- `nn.GRU(16, nb_unites, num_layers=2, batch_first=True)` : la taille des vecteurs d'entrée d'abord ; `batch_first=True` pour des tenseurs de forme `(batch, longueur, …)`
- un `nn.GRU` renvoie deux valeurs : sa sortie à chaque pas, puis `h_n`, le dernier état caché de chaque couche, de forme `(couches, batch, nb_unites)` ; `h_n[-1]`, celui de la dernière couche, est de forme `(batch, nb_unites)`
- `resume.unsqueeze(1).repeat(1, longueur_max_resultat, 1)` : de `(batch, nb_unites)` à `(batch, 6, nb_unites)`, le résumé répété 6 fois
- le décodeur : `nn.GRU(nb_unites, nb_unites, batch_first=True)`, dont on garde la sortie à chaque pas, de forme `(batch, 6, nb_unites)`
- `nn.Linear(nb_unites, len(caracteres))` agit sur la dernière dimension d'un tenseur, donc sur chaque pas séparément : appliquée à la sortie du décodeur, elle donne la forme `(batch, 6, 16)`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
class Seq2Seq(nn.Module):
  def __init__(self, nb_unites: int = 512) -> None:
    super().__init__()
    self.embedding = nn.Embedding(len(caracteres), ...)
    self.encodeur = nn.GRU(..., nb_unites, num_layers=..., batch_first=True)
    self.decodeur = nn.GRU(..., nb_unites, batch_first=True)
    self.tete = nn.Linear(nb_unites, ...)

  def forward(self, equations: torch.Tensor) -> torch.Tensor:
    _, etats = self.encodeur(self.embedding(equations))
    resume = etats[...]
    resume_repete = resume.unsqueeze(1).repeat(1, ..., 1)
    sorties_decodeur, _ = self.decodeur(resume_repete)
    return self.tete(...)
```

### Solution

In [ ]:
class Seq2Seq(nn.Module):
  """Encode l'équation, puis décode le résultat caractère par caractère."""

  def __init__(self, nb_unites: int = 512) -> None:
    super().__init__()
    # Encodeur : deux couches de GRU empilées
    self.embedding = nn.Embedding(len(caracteres), 16)
    self.encodeur = nn.GRU(16, nb_unites, num_layers=2, batch_first=True)
    # Décodeur : un GRU, puis un score par caractère à chaque pas
    self.decodeur = nn.GRU(nb_unites, nb_unites, batch_first=True)
    self.tete = nn.Linear(nb_unites, len(caracteres))

  def forward(self, equations: torch.Tensor) -> torch.Tensor:
    _, etats = self.encodeur(self.embedding(equations))
    # Dernier état caché de la dernière couche : le résumé de l'équation,
    # répété pour chacun des caractères du résultat
    resume = etats[-1]
    resume_repete = resume.unsqueeze(1).repeat(1, longueur_max_resultat, 1)
    sorties_decodeur, _ = self.decodeur(resume_repete)
    # Appliquée à chaque pas : (batch, 6, 16)
    return self.tete(sorties_decodeur)

### Le modèle

La cellule suivante enveloppe un `Seq2Seq` dans un `PredicteurSequences`, un `LightningModule` : la perte d'entropie croisée et l'exactitude, calculées sur chacun des 6 caractères du résultat dans sa méthode `_step`, l'optimiseur Adam. `predit` applique un modèle à des équations, batch par batch. La cellule construit ensuite le modèle du TP et en affiche le résumé.

In [ ]:
class PredicteurSequences(lightning.LightningModule):
  """Enveloppe Lightning : une classe prédite pour chaque caractère de sortie."""

  def __init__(self, modele: nn.Module, taux_apprentissage: float = 1e-3) -> None:
    super().__init__()
    self.modele = modele
    self.taux_apprentissage = taux_apprentissage
    # Une entrée d'exemple : le résumé du modèle affiche les formes des sorties
    self.example_input_array = torch.zeros(1, longueur_max_equation, dtype=torch.long)

  def forward(self, equations: torch.Tensor) -> torch.Tensor:
    return self.modele(equations)

  def _step(self, batch: tuple[torch.Tensor, torch.Tensor], etape: str) -> torch.Tensor:
    equations, resultats = batch
    # Logits de forme (batch, 6, 16) : la perte attend la dimension des classes
    # en deuxième position
    logits = self(equations).transpose(1, 2)
    perte = nn.functional.cross_entropy(logits, resultats)
    # Exactitude par caractère : un résultat à un caractère faux compte pour 5/6
    exactitude = (logits.argmax(dim=1) == resultats).float().mean()
    # Moyennes de chaque epoch, pondérées par la taille des batchs
    self.log_dict(
      {f"perte_{etape}": perte, f"exactitude_{etape}": exactitude},
      on_step=False,
      on_epoch=True,
      prog_bar=True,
    )
    return perte

  def training_step(
    self, batch: tuple[torch.Tensor, torch.Tensor], indice_lot: int
  ) -> torch.Tensor:
    return self._step(batch, "entrainement")

  def validation_step(
    self, batch: tuple[torch.Tensor, torch.Tensor], indice_lot: int
  ) -> None:
    self._step(batch, "val")

  def configure_optimizers(self) -> torch.optim.Optimizer:
    return torch.optim.Adam(self.parameters(), lr=self.taux_apprentissage)


@torch.no_grad()
def predit(modele: nn.Module, X: torch.Tensor, taille_lot: int = 1024) -> torch.Tensor:
  """Caractère le plus probable à chaque pas, batch par batch."""
  modele.eval()
  device = next(modele.parameters()).device
  return torch.cat(
    [modele(batch.to(device)).argmax(dim=-1).cpu() for batch in X.split(taille_lot)]
  )


modele = PredicteurSequences(Seq2Seq())
print(ModelSummary(modele, max_depth=2))

## 4. Entraîner, puis évaluer le modèle

La fonction `entraine` entraîne un modèle avec Lightning, sur des batchs de 512 équations, et renvoie ses métriques de chaque epoch, lues dans le journal du `CSVLogger`. Les équations de test servent à la validation.

Pendant l'entraînement, le callback `AfficheExemples` affiche toutes les 10 epochs les résultats que le modèle donne pour les 20 mêmes équations de test : on le voit progresser. L'entraînement dure 30 epochs : au-delà, la perte de validation ne baisse plus, alors que celle d'entraînement baisse encore.

L'exactitude que Lightning affiche est celle des caractères : un résultat dont un seul caractère est faux y compte pour 5/6.

In [ ]:
def entraine(
  modele: PredicteurSequences,
  donnees: tuple[torch.Tensor, ...],
  nom: str,
  max_epoques: int,
  callbacks: Sequence[Callback] = (),
) -> pandas.DataFrame:
  X_train, X_val, y_train, y_val = donnees
  trainer = lightning.Trainer(
    max_epochs=max_epoques,
    accelerator="auto",
    devices=1,
    logger=CSVLogger("logs", name=nom),
    # La barre de progression de tqdm : celle de rich, que Lightning choisit
    # quand rich est installé (Colab), ne laisse pas un callback écrire avec print
    callbacks=[TQDMProgressBar(), *callbacks],
    enable_checkpointing=False,
    enable_model_summary=False,
  )
  trainer.fit(
    modele,
    DataLoader(TensorDataset(X_train, y_train), batch_size=512, shuffle=True),
    DataLoader(TensorDataset(X_val, y_val), batch_size=2048),
  )
  # Métriques de chaque epoch, écrites par le CSVLogger sur des lignes
  # différentes pour l'entraînement et la validation : on les regroupe
  return (
    pandas.read_csv(pathlib.Path(trainer.logger.log_dir) / "metrics.csv")
    .groupby("epoch")
    .mean(numeric_only=True)
  )


X_exemples, y_exemples = X_test[:20], y_test[:20]


def affiche_exemples(modele: nn.Module) -> None:
  propositions = predit(modele, X_exemples)
  for equation, resultat, proposition in zip(X_exemples, y_exemples, propositions):
    print(f"{decode(equation.flip(0))} = {decode(resultat)} ?= {decode(proposition)}")


class AfficheExemples(Callback):
  def __init__(self, periode: int = 10) -> None:
    super().__init__()
    self.periode = periode

  def on_train_epoch_end(
    self, trainer: lightning.Trainer, pl_module: lightning.LightningModule
  ) -> None:
    epoque = trainer.current_epoch + 1
    if epoque % self.periode == 0:
      print(f"\nAprès {epoque} epochs :")
      affiche_exemples(pl_module)  # le modèle que le Trainer entraîne
      pl_module.train()  # predit l'a mis en mode évaluation


NB_EPOQUES = 30
historique = entraine(
  modele,
  (X_train, X_test, y_train, y_test),
  "seq2seq",
  max_epoques=NB_EPOQUES,
  callbacks=[AfficheExemples(periode=10)],
)

La fonction `evalue` mesure, sur les équations de test :

- la proportion de prédictions illégales, qui ne se lisent pas comme un nombre (`1.2.3`) ;
- l'exactitude des résultats entiers : la proportion de résultats exacts, au caractère près ;
- l'erreur relative $\frac{y_{pred} - y}{y}$ des prédictions légales, dont elle trace la distribution.

In [ ]:
def evalue(modele: nn.Module) -> None:
  predictions = predit(modele, X_test)
  exact = (predictions == y_test).all(dim=1)

  # Les prédictions qui se lisent comme un nombre, et leurs vrais résultats
  y_pred, y_true = [], []
  for prediction, resultat in zip(predictions, y_test):
    try:
      y_pred.append(float(decode(prediction)))
    except ValueError:
      continue
    y_true.append(float(decode(resultat)))
  y_pred, y_true = np.array(y_pred), np.array(y_true)
  illegales = 1 - len(y_pred) / len(y_test)
  print(f"Prédictions illégales : {illegales:.1%}")
  print(f"Résultats exacts : {exact.float().mean():.1%}")
  # Un modèle pas encore entraîné peut n'avoir aucune prédiction légale
  if len(y_pred) == 0:
    return

  # Erreur relative, divisée par 1 quand le vrai résultat est nul
  erreur_relative = (y_pred - y_true) / np.where(y_true == 0, 1, y_true)
  print(f"Erreur relative absolue médiane : {np.median(np.abs(erreur_relative)):.2%}")

  fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
  # Des classes régulières sur l'axe logarithmique ; un résultat exact compte à 1e-4
  erreurs = np.maximum(np.abs(erreur_relative), 1e-4)
  ax1.hist(erreurs, bins=np.geomspace(1e-4, erreurs.max(), 50))
  ax1.set_xscale("log")
  ax1.set_yscale("log")
  ax1.set_title("Toutes les erreurs relatives, en valeur absolue")
  ax2.hist(erreur_relative, bins=20, range=(-0.1, 0.1))
  ax2.set_title("Zoom autour de 0")
  for ax in (ax1, ax2):
    ax.set_xlabel("Erreur relative")
    ax.set_ylabel("Équations")
  plt.show()


evalue(modele)

## 5. La réussite par opération

Le modèle réussit-il aussi bien les 4 opérations ? Calculez, pour chacune, la proportion d'équations de test dont le résultat prédit est **entièrement** juste, et expliquez les écarts.

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `predit(modele, X_test)` : le caractère le plus probable à chaque pas, de forme `(20000, 6)`
- `(predictions == y_test).all(dim=1)` : vrai pour un résultat dont les 6 caractères sont justes
- `decode(equation.flip(0))` remet une équation dans l'ordre, et `"+" in equation` dit si c'est une addition
- un tenseur booléen sélectionne des lignes : `exact[masque]` ; `.float().mean()` donne la proportion de `True`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
predictions = predit(modele, X_test)
exact = (predictions == ...).all(dim=...)
equations = [decode(equation.flip(0)) for equation in X_test]
for operation in operations:
  masque = torch.tensor([operation in equation for equation in ...])
  print(f"{operation} : {exact[...].float().mean():.1%}")
```

### Solution

In [ ]:
predictions = predit(modele, X_test)
# Un résultat n'est juste que si ses 6 caractères le sont
exact = (predictions == y_test).all(dim=1)
equations = [decode(equation.flip(0)) for equation in X_test]
for operation in operations:
  masque = torch.tensor([operation in equation for equation in equations])
  print(f"{operation} : {exact[masque].float().mean():.1%} de résultats exacts")

Les écarts sont énormes : l'addition et la soustraction sont presque toujours justes (94 % de résultats exacts), la multiplication et la division presque jamais (moins de 1 %). Dans une addition ou une soustraction, chaque chiffre du résultat ne dépend que de quelques chiffres des opérandes et d'une retenue. Dans une multiplication, chaque chiffre du résultat dépend de tous ceux des opérandes, et une division donne souvent des décimales sans structure simple : ces deux opérations sont bien plus difficiles.

Le modèle n'y est pourtant pas perdu : il trouve souvent l'ordre de grandeur et les premiers chiffres (dans notre exécution, `721/866 = 0.8325` est prédit `0.8322`, et `97*727 = 70519`, `70319`), rarement tous. Le zoom de la partie 4 le montre : en dehors des résultats exacts, la plupart des erreurs relatives restent sous 10 %.

## 6. Faut-il inverser l'équation ?

`fabrique_probleme` inverse les équations par défaut : l'encodeur lit `321+654` sous la forme `456+123`. Générez un jeu de données sans inversion, entraînez le même modèle dessus pendant 30 epochs, et comparez sa courbe d'exactitude de validation à celle du premier modèle. Comment expliquer l'écart ?

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `fabrique_probleme(100_000, inverse=False)` : les équations dans l'ordre
- `sklearn.model_selection.train_test_split` renvoie les 4 tenseurs dans l'ordre qu'attend le paramètre `donnees` de `entraine` : `(X_train, X_test, y_train, y_test)`
- `historique["exactitude_val"]` : l'exactitude de validation de chaque epoch du premier modèle ; `[:30]` garde ses 30 premières

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
X_brut, y_brut = fabrique_probleme(100_000, inverse=...)
donnees_brutes = sklearn.model_selection.train_test_split(
  X_brut, y_brut, train_size=0.8, random_state=0
)
historique_brut = entraine(
  PredicteurSequences(Seq2Seq()), ..., "seq2seq_brut", max_epoques=...
)

plt.plot(historique["exactitude_val"][:30], label=...)
plt.plot(historique_brut[...], label=...)
plt.legend()
plt.show()
```

### Solution

In [ ]:
X_brut, y_brut = fabrique_probleme(100_000, inverse=False)
donnees_brutes = sklearn.model_selection.train_test_split(
  X_brut, y_brut, train_size=0.8, random_state=0
)
historique_brut = entraine(
  PredicteurSequences(Seq2Seq()), donnees_brutes, "seq2seq_brut", max_epoques=30
)

plt.plot(historique["exactitude_val"][:30], label="Équation inversée")
plt.plot(historique_brut["exactitude_val"], label="Équation dans l'ordre")
plt.xlabel("Epoch (à partir de 0)")
plt.ylabel("Exactitude de validation (par caractère)")
plt.legend()
plt.show()

Les deux modèles progressent au même rythme pendant les 7 premières epochs, puis celui des équations inversées prend nettement l'avantage : après 10 epochs, 63 % des caractères de validation sont justes, contre 59 % dans l'ordre ; après 15, 74 % contre 64 %. Sur les équations dans l'ordre, le modèle rattrape ensuite une partie de son retard, sans le combler : 71 % contre 76 % à la 30e epoch.

Les premiers caractères du résultat dépendent surtout des premiers chiffres des opérandes, en particulier de ceux de l'opérande de gauche. Inversée, l'équation commence par l'opérande de droite et finit par celui de gauche : l'encodeur lit ces premiers chiffres en dernier, ils sont plus présents dans son dernier état caché, qui est tout ce que reçoit le décodeur. C'est l'astuce de l'article [*Sequence to Sequence Learning with Neural Networks*](https://arxiv.org/abs/1409.3215), qui inversait les phrases sources en traduction automatique.